# **Fine-tuning de Gemma 3 con QLoRA**

En este apartado realizamos el fine-tuning del modelo de lenguaje Gemma 3 4B
para que genere explicaciones clínicas en español sobre los resultados de la CNN.

Usamos QLoRA (Quantized Low-Rank Adaptation), una técnica que permite
fine-tunear modelos grandes de forma eficiente en GPU con poca memoria.
En lugar de actualizar todos los parámetros del modelo, QLoRA añade matrices
de bajo rango (LoRA) solo en las capas de atención, reduciendo drásticamente
el número de parámetros entrenables.

El modelo base elegido es `google/gemma-3-4b-it` porque:
- Es open-source y gratuito
- Al ser instruction-tuned ya tiene una base sólida para seguir instrucciones
- El tamaño 4B permite fine-tuning eficiente en Colab con QLoRA en 4-bit
- Ofrece un buen equilibrio entre capacidad de generación y consumo de memoria

In [ ]:
# Instalación de dependencias
%pip install -q -U transformers accelerate peft bitsandbytes datasets trl sentencepiece

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

import os
#os.chdir("/content/drive/MyDrive/Colab Notebooks/ECGAssistant_Sprint1")
print("Directorio:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directorio: /content/drive/.shortcut-targets-by-id/1acOp95JDTEWtgyHkn2BDLpHG-oeo8sXh/ECGAssistant_Sprint1


In [ ]:
from huggingface_hub import login
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# Login a Hugging Face
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

## 1. Carga y preparación del dataset

Cargamos el dataset de preguntas y respuestas clínicas en formato JSONL.
Cada ejemplo tiene tres partes: system (rol del asistente), user (pregunta)
y assistant (respuesta esperada). Los concatenamos en un texto único que
será el input del entrenamiento.

In [ ]:
# Cargamos el dataset desde el archivo JSONL
dataset = load_dataset("json", data_files="ecg_qa_dataset.jsonl")
dataset = dataset["train"]

def format_example(example):
    """Convierte el formato messages a texto plano para el entrenamiento."""
    messages = example["messages"]

    system_msg    = next((m["content"] for m in messages if m["role"] == "system"), "")
    user_msg      = next((m["content"] for m in messages if m["role"] == "user"), "")
    assistant_msg = next((m["content"] for m in messages if m["role"] == "assistant"), "")

    return {
        "text": f"{system_msg}\n\n### Instrucción:\n{user_msg}\n\n### Respuesta:\n{assistant_msg}"
    }

# Aplicamos el formato y hacemos el split 90/10
dataset = dataset.map(format_example, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset)
print("\nEjemplo de entrada:")
print(dataset["train"][0]["text"][:500])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 180
    })
    test: Dataset({
        features: ['text'],
        num_rows: 20
    })
})

Ejemplo de entrada:
Eres ECGAssistant, un asistente médico especializado en la interpretación de electrocardiogramas (ECG). Tu función es explicar los resultados de un modelo de deep learning (CNN) que clasifica señales ECG como normales o anormales.

Reglas:
1. Siempre basa tu respuesta en el resultado de la CNN proporcionado por el usuario.
2. Explica de forma clara y precisa el significado clínico del resultado.
3. Cuando sea relevante, menciona posibles arritmias, estudios complementarios o recomendaciones.
4. 


## 2. Configuración del modelo en 4-bit

Cargamos Gemma 3 4B cuantizado en 4-bit con la configuración NF4
(Normal Float 4), que reduce el consumo de VRAM de ~16GB a ~5GB
sin degradar de forma crítica la calidad del modelo.

La doble cuantización (double_quant) aplica una segunda cuantización
sobre los propios factores de escala, ahorrando memoria adicional.

In [ ]:
model_id    = "google/gemma-3-4b-it"
output_dir  = "./gemma3_qlora"

# Configuración de cuantización en 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Cargamos tokenizer y modelo
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache         = False
model.config.use_token_type_ids = True
model.config.type_vocab_size    = 1

print("Modelo cargado correctamente")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Modelo cargado correctamente


## 3. Configuración de LoRA

LoRA añade matrices de bajo rango en las capas de atención del transformer.
En lugar de actualizar todos los parámetros del modelo (unos 4B parámetros), solo entrenamos las matrices LoRA (unos 3M parámetros), lo que hace el fine-tuning mucho más rápido y eficiente.

Los parámetros elegidos:
- `r=8`: rango de las matrices LoRA. Valor bajo para reducir VRAM manteniendo
  capacidad de adaptación.
- `lora_alpha=16`: escala de las actualizaciones LoRA.
- `target_modules`: aplicamos LoRA solo en las capas de atención de Gemma.
- `lora_dropout=0.05`: regularización para evitar overfitting.

In [ ]:
# Preparamos el modelo para entrenamiento en 4-bit
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 5,949,440 || all params: 4,306,028,912 || trainable%: 0.1382


## 4. Tokenización del dataset

Tokenizamos los ejemplos con una longitud máxima de 768 tokens.
Usamos padding fijo para mantener batches uniformes y controlar
el consumo de memoria.

In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=768,
    )
    # Gemma 3 no usa token_type_ids pero lo añadimos para compatibilidad
    tokens["token_type_ids"] = [0] * len(tokens["input_ids"])
    return tokens

tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names,
)

print("Dataset tokenizado:")
print(tokenized_dataset)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset tokenizado:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'token_type_ids'],
        num_rows: 180
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'token_type_ids'],
        num_rows: 20
    })
})


## 5. Entrenamiento

Usamos batch size de 1 con gradient accumulation de 16 pasos, lo que
equivale a un batch efectivo de 16 sin necesitar esa memoria de golpe.
El optimizador paged_adamw_8bit gestiona la memoria de forma eficiente
paginando los estados del optimizador a CPU cuando no caben en GPU.

In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    max_grad_norm=1.0,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.911215
20,1.329453
30,1.014898


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=36, training_loss=1.3379662831624348, metrics={'train_runtime': 1047.1329, 'train_samples_per_second': 0.516, 'train_steps_per_second': 0.034, 'total_flos': 9032745524428800.0, 'train_loss': 1.3379662831624348, 'epoch': 3.0})

## 6. Guardado del modelo fine-tuneado

Guardamos los pesos LoRA y el tokenizer para poder cargarlos en el
notebook del pipeline sin necesidad de reentrenar.

In [ ]:
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Modelo guardado en {output_dir}")

Modelo guardado en ./gemma3_qlora


## 7. Inferencia de prueba

Probamos el modelo fine-tuneado con algunas preguntas clínicas para
verificar que genera respuestas coherentes en español.

In [ ]:
# Preparamos el modelo para inferencia
model.eval()
model.config.use_cache = True

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
if hasattr(model.base_model, "gradient_checkpointing_disable"):
    model.base_model.gradient_checkpointing_disable()


def ask(question):
    """Genera una respuesta clínica a partir de una pregunta."""
    prompt = f"### Instrucción:\n{question}\n\n### Respuesta:\n"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True)

    if "token_type_ids" not in inputs:
        inputs["token_type_ids"] = torch.zeros_like(inputs["input_ids"])

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            use_cache=True
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# Pruebas de inferencia
preguntas = [
    "¿Qué es la fibrilación ventricular?",
    "Explica la taquicardia ventricular en términos sencillos.",
    "¿Qué complicaciones puede causar la fibrilación auricular?",
    "¿Qué se observa en un ECG cuando hay bloqueo AV?"
]

for pregunta in preguntas:
    print(f"Pregunta: {pregunta}")
    print(f"Respuesta: {ask(pregunta)}")
    print("-" * 60)

Pregunta: ¿Qué es la fibrilación ventricular?
Respuesta: ### Instrucción:
¿Qué es la fibrilación ventricular?

### Respuesta:
La fibrilación ventricular (FV) es un tipo de arritmia cardíaca que se caracteriza por un latido cardíaco irregular y caótico. En lugar de bombear sangre de manera eficiente, el corazón se contrae y se desenrolla de forma incontrolada. Esto significa que no hay un ritmo organizado que impulse la sangre hacia el cuerpo.

Aquí hay algunos puntos clave sobre la FV:

*   **Mecanismo:** La actividad eléctrica del corazón se descontrola, provocando contracciones aleatorias e ineficaces.
*   **Consecuencias:** La FV no suministra oxígeno a los órganos vitales, lo que puede llevar a la pérdida de conciencia y, en cuestión de minutos, a la muerte.
*   **Tratamiento:** El tratamiento principal es la desfibrilación eléctrica (administración de una descarga eléctrica para restablecer el ritmo normal), aunque también se pueden usar medicamentos y otros procedimientos.

Es im

## Reflexión

El fine-tuning con QLoRA permite adaptar un modelo de 4B parámetros en
Colab gratuito en pocas horas. Los resultados muestran que el modelo
genera explicaciones clínicas coherentes en español, ancladas en el
dominio de la cardiología.

El modelo fine-tuneado queda listo para integrarse con la CNN y el sistema
RAG formando el pipeline completo de ECGAssistant.